In [9]:
import numpy as np

# -------------------------------------------------
# Basic 2D Convolution (single channel)
# -------------------------------------------------
def conv2d(X, W, stride=1, padding=0):
    X_padded = np.pad(X, ((padding, padding), (padding, padding)), mode='constant')
    H, W_in = X_padded.shape
    fH, fW = W.shape

    out_H = (H - fH) // stride + 1
    out_W = (W_in - fW) // stride + 1

    out = np.zeros((out_H, out_W))

    for i in range(out_H):
        for j in range(out_W):
            region = X_padded[i*stride:i*stride+fH, j*stride:j*stride+fW]
            out[i, j] = np.sum(region * W)

    return out

# -------------------------------------------------
# Pooling
# -------------------------------------------------
def max_pool(X, size=2, stride=2):
    H, W = X.shape
    out_H = (H - size) // stride + 1
    out_W = (W - size) // stride + 1

    out = np.zeros((out_H, out_W))

    for i in range(out_H):
        for j in range(out_W):
            region = X[i*stride:i*stride+size, j*stride:j*stride+size]
            out[i, j] = np.max(region)

    return out

def avg_pool(X, size=2, stride=2):
    H, W = X.shape
    out_H = (H - size) // stride + 1
    out_W = (W - size) // stride + 1

    out = np.zeros((out_H, out_W))

    for i in range(out_H):
        for j in range(out_W):
            region = X[i*stride:i*stride+size, j*stride:j*stride+size]
            out[i, j] = np.mean(region)

    return out


##(a) Given the input data X and the filter W of one convolutional layer, write the outputs y0 when the stride is set as S=1 and S=2.


In [10]:
X = np.array([
    [1.23, 0.94, 0.83, 0.2],
    [0.34, 0.44, 0.37, 0.3],
    [0.34, 1.14, 2.34, 1.24],
    [0.49, 0.18, 2.04, 1.42]
])

W = np.array([
    [1, 0.5],
    [0.3, 1]
])

print("=== (a) S=1 ===")
out_s1 = conv2d(X, W, stride=1, padding=0)
print(out_s1)

print("\n=== (a) S=2 ===")
out_s2 = conv2d(X, W, stride=2, padding=0)
print(out_s2)


=== (a) S=1 ===
[[2.242 1.857 1.341]
 [1.802 3.307 2.462]
 [1.237 4.404 4.992]]

=== (a) S=2 ===
[[2.242 1.341]
 [1.237 4.992]]


##(b) To prevent the change of the data size, we use zero-padding and set P=1. Recalculate the outputs y0 in (a). With the stride is set as S=1, write the outputs when the y0 is processed by max-pooling and average-pooling.

In [11]:
# -------------------------------------------------
# (b) padding=1, S=1
# -------------------------------------------------
print("\n=== (b) Padding=1, S=1 ===")
out_pad = conv2d(X, W, stride=1, padding=1)
print(out_pad)

print("\nMax Pooling, S=1:")
print(max_pool(out_pad, size=2, stride=1))

print("\nAverage Pooling, S=1:")
print(avg_pool(out_pad, size=2, stride=1))


=== (b) Padding=1, S=1 ===
[[1.23  1.309 1.112 0.449 0.06 ]
 [0.955 2.242 1.857 1.341 0.29 ]
 [0.51  1.802 3.307 2.462 0.672]
 [0.66  1.237 4.404 4.992 1.666]
 [0.245 0.58  1.2   2.75  1.42 ]]

Max Pooling, S=1:
[[2.242 2.242 1.857 1.341]
 [2.242 3.307 3.307 2.462]
 [1.802 4.404 4.992 4.992]
 [1.237 4.404 4.992 4.992]]

Average Pooling, S=1:
[[1.434   1.63    1.18975 0.535  ]
 [1.37725 2.302   2.24175 1.19125]
 [1.05225 2.6875  3.79125 2.448  ]
 [0.6805  1.85525 3.3365  2.707  ]]


In [12]:
# -------------------------------------------------
# (c) Multi-channel convolution
# -------------------------------------------------

def conv2d_multi_channel(X, filters, stride=1, padding=0):
    """
    X: (C, H, W)
    filters: (N, C, fH, fW)
    """
    C, H, W_in = X.shape
    N, _, fH, fW = filters.shape

    X_padded = np.pad(X, ((0,0),(padding,padding),(padding,padding)), mode='constant')
    H_p, W_p = X_padded.shape[1:]

    out_H = (H_p - fH)//stride + 1
    out_W = (W_p - fW)//stride + 1

    out = np.zeros((N, out_H, out_W))

    for n in range(N):
        for i in range(out_H):
            for j in range(out_W):
                region = X_padded[:, i*stride:i*stride+fH, j*stride:j*stride+fW]
                out[n, i, j] = np.sum(region * filters[n])

    return out


# RGB input
R = np.array([
    [123, 94, 83],
    [34, 44, 37],
    [34, 114, 234]
])

G = np.array([
    [123, 94, 20],
    [11, 30, 22],
    [12, 40, 23]
])

B = np.array([
    [123, 94, 83],
    [34, 44, 37],
    [34, 114, 234]
])

X_rgb = np.stack([R, G, B])  # shape (3,3,3)

W1 = np.array([
    [1, 0.5],
    [0.3, 1]
])

W2 = np.array([
    [0.5, 1],
    [1, 0.3]
])

# Expand filters to 3 channels (same kernel applied to each channel)
filters = np.stack([
    np.stack([W1, W1, W1]),
    np.stack([W2, W2, W2])
])

print("\n=== (c) Multi-channel, P=1, S=1 ===")
out_multi = conv2d_multi_channel(X_rgb, filters, stride=1, padding=1)
print("Output shape:", out_multi.shape)
print(out_multi)


=== (c) Multi-channel, P=1, S=1 ===
Output shape: (2, 4, 4)
[[[369.  392.7 270.6  55.8]
  [263.5 651.7 506.4 214.8]
  [119.5 430.  737.4 243.3]
  [ 40.  214.  513.5 491. ]]

 [[110.7 453.6 337.8 186. ]
  [392.7 580.9 473.8 189. ]
  [103.  317.9 570.3 539. ]
  [ 80.  308.  625.  245.5]]]
